# ✅ Импорты

In [1]:
import os

import numpy as np
import pandas as pd

import hyperopt.hp as hp

from sklearn.model_selection import train_test_split

from catboost import CatBoostClassifier

from sklift.datasets import fetch_hillstrom
from sklift.metrics import qini_auc_score, uplift_at_k

# econml
from econml.dml import CausalForestDML

#causalml
from causalml.inference.tree import UpliftRandomForestClassifier

from upninja.dml.UpliftRandomForestDML import UpliftRandomForestDML
from upninja.dml.UpliftTreeClassifierDML import UpliftTreeClassifierDML
from upninja.tune.Selection import UpliftTune

import matplotlib.pyplot as plt
import seaborn as sns

/Users/romanseleznyov/Documents/UpliftNinja/uplift-env/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/romanseleznyov/Documents/UpliftNinja/uplift-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to import duecredit due to No module named 'duecredit'


In [2]:
SEED = 42

# ✅ Fetch & Preproc

In [3]:
data = fetch_hillstrom(target_col='visit')
df = data.data
X, y, w = df, data.target, data.treatment

# Hillstrom: treatment — 'Mens E-Mail', 'Womens E-Mail' -> объединим в бинарный treatment
w = (w != 'No E-Mail').astype(int)  # 1 — получили email, 0 — нет

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=SEED
)

num_cols = X_train.select_dtypes(exclude='object').columns
X_train = X_train[num_cols]
X_test = X_test[num_cols]

# for causalml
X_np = X[num_cols].values
y_np = y.values
X_train_np, X_test_np, y_train_np, y_test_np, w_train_str, w_test_str = (
    X_train.values, X_test.values,
    y_train.values, y_test.values,
    np.where(w_train == 1, 'treatment', 'control'),
    np.where(w_test == 1, 'treatment', 'control')
)

models_results = {
    'approach': [],
    'uplift@30%': [],
    'qini_auc': []
}

In [4]:
df.shape

(64000, 8)

# ✅ EconML

In [5]:
%%time

eml_params = {
    'n_estimators': hp.choice('n_estimators', [n for n in range(10, 100, 2)]),
    'max_depth': hp.choice('depth', [2, 4, 6, 8, 10]),
    'min_samples_split': hp.uniformint('min_samples_split', 10, 200),
    'subforest_size': 2,
    'n_jobs': -1,
    'random_state': SEED
}

eml_tune = UpliftTune(
    uplift_model_class=CausalForestDML,
    data=X_train,
    target=y_train,
    treatment=w_train,
    space=eml_params,
    verbose=True,
    max_evals=5
)

eml_tuned = eml_tune.tune()['best_params']

100%|█████████| 5/5 [04:04<00:00, 48.86s/trial, best loss: -0.08009707207053174]
Optimization completed. Best score: 0.0801
Best parameters: {'max_depth': 4, 'min_samples_split': 126, 'n_estimators': 86, 'n_jobs': -1, 'random_state': 42, 'subforest_size': 2}
CPU times: user 4min 10s, sys: 2.34 s, total: 4min 12s
Wall time: 4min 4s


In [9]:
%%time
rf_eml = CausalForestDML(
    **eml_tuned
)

rf_eml.fit(Y=y_train, T=w_train, X=X_train)
uplift_eml = rf_eml.effect(X_test)

eml_score = uplift_at_k(y_true=y_test, uplift=uplift_eml, treatment=w_test, strategy='by_group', k=0.3)
eml_score_q = qini_auc_score(y_true=y_test, uplift=uplift_eml, treatment=w_test)

models_results['approach'].append('EconML')
models_results['uplift@30%'].append(eml_score)
models_results['qini_auc'].append(eml_score_q)

CPU times: user 27.9 s, sys: 236 ms, total: 28.1 s
Wall time: 26.6 s


# ✅ CausalML

In [10]:
%%time

rf_cm = UpliftRandomForestClassifier(
    n_estimators=50,
    min_samples_leaf=100,
    max_depth=4,
    random_state=SEED,
    control_name='control'
)

rf_cm.fit(
    X=X_train_np,
    treatment=w_train_str,
    y=y_train_np
)

rf_cm.fit(X_train_np, w_train_str, y_train_np)
uplift_cm = rf_cm.predict(X_test_np).flatten()

cm_score = uplift_at_k(y_true=y_test_np, uplift=uplift_cm, treatment=w_test, strategy='by_group', k=0.3)
cm_score_q = qini_auc_score(y_true=y_test_np, uplift=uplift_cm, treatment=w_test)

models_results['approach'].append('CausalML')
models_results['uplift@30%'].append(cm_score)
models_results['qini_auc'].append(cm_score_q)

CPU times: user 5.12 s, sys: 839 ms, total: 5.96 s
Wall time: 3.21 s


# ✅ UpNinja

In [11]:
%%time

space = {
    'min_samples': hp.uniformint('min_samples', 20, 500),
    'max_depth': hp.uniformint('max_depth', 3, 20),
    'bins': hp.uniformint('bins', 5, 30),
    'min_samples_treatment': hp.uniformint("min_samples_treatment", 10, 100),
    'random_state': SEED
}

dml_tune = UpliftTune(
    uplift_model_class=UpliftTreeClassifierDML,
    data=X_train,
    target=y_train,
    treatment=w_train,
    space=space,
    verbose=True
)

dml_tuned = dml_tune.tune()['best_params']

100%|███████| 50/50 [00:07<00:00,  6.49trial/s, best loss: -0.07361440689763528]
Optimization completed. Best score: 0.0736
Best parameters: {'bins': 27, 'max_depth': 5, 'min_samples': 244, 'min_samples_treatment': 16, 'random_state': 42}
CPU times: user 7.67 s, sys: 41.9 ms, total: 7.71 s
Wall time: 7.71 s


In [12]:
%%time
rf_un = UpliftRandomForestDML(
    **dml_tuned
)

rf_un.fit(X_train, y_train, w_train)
uplift_un = rf_un.predict(X_test)

un_score = uplift_at_k(y_true=y_test, uplift=uplift_un, treatment=w_test, strategy='by_group', k=0.3)
un_score_q = qini_auc_score(y_true=y_test, uplift=uplift_un, treatment=w_test)

models_results['approach'].append('UpninjaRF')
models_results['uplift@30%'].append(un_score)
models_results['qini_auc'].append(un_score_q)

CPU times: user 195 ms, sys: 3.47 ms, total: 198 ms
Wall time: 197 ms


# ✅ Результаты

In [13]:
pd.DataFrame(data=models_results).sort_values(['uplift@30%', 'qini_auc'], ascending=False)

,approach,uplift@30%,qini_auc
0,EconML,0.091372,0.057087
2,UpninjaRF,0.084424,0.033440
1,CausalML,0.066734,0.029050
